# 08 · 采样策略：greedy / temperature / top-k / top-p

> **学习目标**：把 LLM 推理时「同样输入为什么输出每次不一样」搞清楚。手撸 4 种采样、再用本机 Qwen 验证。
>
> **预备**：04 + 05 已过；想跑 Part B 需要先在另一个终端启动 `ollama serve`。
>
> **为什么重要**：调 API 时一两个参数错就答非所问；本地部署时采样策略直接决定「答案稳定 vs 富有创意」的取舍。

**本 notebook 分两部分**：
- **Part A**：用合成 logits 手撸 4 种采样 —— **不依赖 Ollama，永远能跑**
- **Part B**：用本机 Qwen 真实验证 —— 需要 Ollama

In [ ]:
import torch, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

torch.manual_seed(0)
np.random.seed(0)

# Part A · 用合成 logits 手撸采样

假设我们有一个 vocab=10、当前位置的 logits 已经算出来了。下面看 4 种策略各自怎么选下一个 token。

In [ ]:
# 假 vocab 与对应「词」
vocab = ['the', 'a', 'cat', 'dog', 'sat', 'ran', 'on', 'mat', 'rug', 'and']
# 假 logits（数值越大越倾向被选）
logits = torch.tensor([3.5, 2.8, 1.2, 1.0, 0.5, 0.3, 0.2, 0.1, -0.5, -1.0])

def show_dist(probs, title):
    plt.figure(figsize=(8, 2.5))
    plt.bar(vocab, probs.numpy())
    plt.title(title); plt.ylim(0, 1); plt.grid(axis='y', alpha=0.3)
    plt.show()

show_dist(logits.softmax(-1), 'baseline softmax (T=1)')

## A1. Greedy — 永远选概率最高的

**特性**：确定性。同样输入永远同样输出。**缺点**：缺少多样性，容易陷入重复（「the the the…」）。

In [ ]:
def greedy_sample(logits):
    return int(logits.argmax())

for _ in range(5):
    idx = greedy_sample(logits)
    print(f'greedy -> {vocab[idx]!r}   (永远是同一个)')

## A2. Temperature — 调整分布尖锐度

$$p_i = \frac{e^{l_i / T}}{\sum_j e^{l_j / T}}$$

- `T < 1`：分布更尖，趋近 greedy
- `T = 1`：原始 softmax
- `T > 1`：分布趋平，更随机
- `T = 0`：数学未定义，工程上 = greedy

In [ ]:
def temperature_sample(logits, T=1.0):
    probs = (logits / T).softmax(-1)
    return int(torch.multinomial(probs, num_samples=1))

for T in [0.3, 0.7, 1.0, 1.5, 3.0]:
    samples = [vocab[temperature_sample(logits, T)] for _ in range(100)]
    top3 = Counter(samples).most_common(3)
    print(f'T={T:.1f}  100 次采样 top-3:', top3)

# 可视化各温度下的分布
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, T in zip(axes, [0.3, 1.0, 2.0, 5.0]):
    ax.bar(vocab, (logits / T).softmax(-1).numpy())
    ax.set_title(f'T = {T}'); ax.tick_params(axis='x', rotation=45); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## A3. Top-k — 只在前 k 个候选里采样

**做法**：把除 top-k 之外的 logits 设为 `-inf`，剩下重做 softmax。

**效果**：避免选到「概率极低的胡言乱语 token」。常用 k=50。

In [ ]:
def top_k_sample(logits, k=5, T=1.0):
    logits = logits.clone() / T
    top_vals, top_idx = torch.topk(logits, k)
    mask = torch.full_like(logits, float('-inf'))
    mask[top_idx] = logits[top_idx]
    probs = mask.softmax(-1)
    return int(torch.multinomial(probs, 1))

for k in [1, 3, 5, 10]:
    samples = [vocab[top_k_sample(logits, k=k)] for _ in range(100)]
    top3 = Counter(samples).most_common(3)
    print(f'top-{k:2d}  100 次:', top3)

## A4. Top-p（nucleus） — 概率累积 ≥ p 的最小集合

**做法**：按概率从高到低排，累加，直到累计 ≥ p（如 0.9），只在这个动态集合里采样。

**优点**：分布平时只考虑少数几个，分布平坦时自动放宽。**现代 LLM 默认推荐 top-p 而非 top-k**。

In [ ]:
def top_p_sample(logits, p=0.9, T=1.0):
    probs = (logits / T).softmax(-1)
    sorted_probs, sorted_idx = probs.sort(descending=True)
    cum = sorted_probs.cumsum(-1)
    # 保留累计概率 <= p 的前缀，至少留 1 个
    keep = cum <= p
    keep[0] = True
    sorted_probs[~keep] = 0
    sorted_probs = sorted_probs / sorted_probs.sum()        # 重新归一
    pick = int(torch.multinomial(sorted_probs, 1))
    return int(sorted_idx[pick])

for p in [0.5, 0.8, 0.95]:
    samples = [vocab[top_p_sample(logits, p=p)] for _ in range(100)]
    print(f'top-p={p}  100 次:', Counter(samples).most_common(5))

### 一张表回看 4 种策略

| 策略 | 确定性 | 多样性 | 何时用 |
|------|-------|-------|--------|
| greedy           | 高（完全确定） | 0 | 抽答案 / 评估 baseline / 工具调用 JSON 输出 |
| temperature      | 中 | 中–高 | 通用对话 / 写作 |
| top-k            | 中 | 中 | 防止极低概率乱码 |
| top-p (nucleus)  | 中 | 中（自适应） | **现代 LLM 推荐默认** |

**实战配方**：`temperature=0` 用于需要稳定 / 抽事实；`temperature=0.7 + top_p=0.9` 用于通用对话；`temperature=1.2 + top_p=0.95` 用于创意写作。

# Part B · 用本机 Qwen 真实验证

**前置条件**：在另一个终端跑 `ollama serve`，且本机至少装了一个 chat 模型（如 `qwen1.5_1.8`、`gemma3:1b`）。

下面用 HTTP API 直接调用，不依赖 ollama python lib（保持最少依赖、最清楚每一步）。

In [ ]:
import requests

BASE = 'http://127.0.0.1:11434'

def ollama_running() -> bool:
    try:
        r = requests.get(f'{BASE}/api/tags', timeout=1)
        return r.status_code == 200
    except Exception:
        return False

if not ollama_running():
    print('❌ Ollama 没在跑。请在另一个终端执行：')
    print('     ollama serve')
    print('   然后回来重跑这个 cell。Part A 已经够你理解所有概念了。')
else:
    tags = requests.get(f'{BASE}/api/tags').json()
    print('✅ Ollama 在线，本机已有模型:')
    for m in tags.get('models', []):
        print(f"  - {m['name']:30}  size {m.get('size',0)/1024/1024:.0f} MB")

In [ ]:
# 把所有你机器上能用的 chat 模型名挑一个填这里
MODEL = 'qwen1.5_1.8'        # 也可换成 'gemma3:1b' 等

def chat(prompt: str, **options) -> str:
    """调用 /api/chat。options 透传给 Ollama，比如 temperature/top_p/top_k/seed。"""
    payload = {
        'model': MODEL,
        'messages': [{'role': 'user', 'content': prompt}],
        'stream': False,
        'options': options,
    }
    r = requests.post(f'{BASE}/api/chat', json=payload, timeout=120)
    r.raise_for_status()
    return r.json()['message']['content']

if ollama_running():
    print(chat('一句话解释 RAG 是什么。', temperature=0))
else:
    print('（Ollama 未在线，跳过实测。）')

In [ ]:
# 同一个 prompt，temperature=0（greedy）跑 5 次 —— 答案应稳定
prompt = '用一句话解释什么是注意力机制。'
if ollama_running():
    print('--- T=0（greedy）应高度一致 ---')
    for i in range(3):
        out = chat(prompt, temperature=0)
        print(f'[{i+1}] {out[:80]}...')
    print('\n--- T=1.5（高随机）应每次不同 ---')
    for i in range(3):
        out = chat(prompt, temperature=1.5, top_p=0.95)
        print(f'[{i+1}] {out[:80]}...')
else:
    print('（Ollama 未在线，跳过。）')

In [ ]:
# 对比 top_p 的影响
if ollama_running():
    prompt2 = '帮我起一个关于「数据飞轮」的产品名。'
    for tp in [0.1, 0.5, 0.95]:
        print(f'\n--- top_p={tp} ---')
        for i in range(3):
            out = chat(prompt2, temperature=1.0, top_p=tp).strip()
            print(f'  [{i+1}] {out[:120]}')
else:
    print('（Ollama 未在线，跳过。）')

## 深入思考

1. **API 同时设 `temperature=0` 和 `top_p=0.5` 会怎样？**
   - `T=0` 优先级高（许多实现直接走 argmax），`top_p` 实质失效。**别同时设这种组合，让意图明确**。
2. **「同一 prompt 不同输出」一定要避免吗？**
   - 否。评估 / 抽事实要稳定（`T=0`）；创意 / 多样性需要随机（`T=0.7~1.0`）。**先想清楚任务再选**。
3. **生产里常见配方**：
   - 工具调用：`T=0`，强制 JSON
   - RAG 问答：`T=0` 或 `T=0.2`，强引用
   - 写作 / Agent 规划：`T=0.7, top_p=0.9`
   - 头脑风暴：`T=1.0~1.5, top_p=0.95`
4. **`seed` 参数能让随机采样可复现吗？**
   - 大多 LLM API 支持 `seed`，**同样 prompt + seed + T > 0 → 可复现**。Ollama 也支持。这是测试时的救命稻草。
5. **温度高了输出就「更聪明」？**
   - 完全不是。温度高只是更随机，**也更容易胡说**。聪明不聪明取决于模型本身。

改一改：在 Part B 的第 4 个 cell 里把 `temperature` 改成 0.0、`seed` 设成 42，跑 3 次，看是否完全一致。

## 自检 ✅

- [ ] 不查文档实现 top-p 采样。
- [ ] 解释 `temperature` 与 `top_p` 同时用时的优先关系。
- [ ] 给一个「答案不稳定」的 LLM 调用，能立刻列出至少 3 个采样参数排查点。
- [ ] 解释「为什么调 API 抽 JSON 时强烈建议 T=0」。
- [ ] 跑过本 notebook Part B 至少一次，能在 ollama 不在线时也讲清概念。

## 下一步

→ [`09_mini_vecdb.ipynb`](09_mini_vecdb.ipynb)